# M2-OE: Melatih M2 agar Tidak Yakin pada Coretan Non-Aksara (Outlier Exposure)
Dijalankan **setelah** M2 selesai dilatih (`hasil_M2/best.keras` sudah ada di Drive). M1 sampai M4 tidak diubah.

1. *Runtime → Change runtime type → **T4 GPU***.
2. *Runtime → **Run all*** (sekitar 10–15 menit).

Hasil tersimpan di folder `hasil_M2OE/` di Drive. Unduh folder itu ke laptop agar aplikasi memakai M2-OE.

## 1. Hubungkan Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os, zipfile

DRIVE = '/content/drive/MyDrive'
NAMA = ['aksara-sunda-resnet50-glcm', 'Preprocessing_ResNet50_GLCM']

def cari_proyek():
    return next((f'{DRIVE}/{n}' for n in NAMA if os.path.exists(f'{DRIVE}/{n}/config.py')), None)

PROJECT_DIR = cari_proyek()
if PROJECT_DIR is None:
    zips = [f'{DRIVE}/{n}.zip' for n in NAMA if os.path.exists(f'{DRIVE}/{n}.zip')]
    assert zips, 'Upload aksara-sunda-resnet50-glcm.zip ke My Drive terlebih dahulu.'
    zipfile.ZipFile(zips[0]).extractall(DRIVE)
    PROJECT_DIR = cari_proyek()

%cd "$PROJECT_DIR"
!pip install -q scikit-image
import tensorflow as tf
print('GPU:', tf.config.list_physical_devices('GPU'))
assert os.path.exists('hasil_M2/best.keras'), 'hasil_M2/best.keras belum ada. Latih M2 dulu.'
print('M2 ditemukan.')

## 2. Tulis skrip ke Drive

In [ ]:
%%writefile buat_outlier.py
"""Membuat contoh citra NON-aksara (outlier) untuk Outlier Exposure.

    python buat_outlier.py            # hasil di folder outlier/{train,val,test}/

Jenis citra: coretan acak, spiral, bentuk geometris, gelombang/zigzag, arsiran,
kata dan angka Latin, serta campurannya. Sebagian dibuat bergaya foto kamera
(latar abu-abu, noise, blur) agar mirip variasi dataset.

Citra yang terlalu mirip aksara asli (korelasi piksel >= MAX_KORELASI terhadap
salah satu citra dataset setelah preprocessing) dibuang, agar model tidak
diajari bahwa bentuk aksara adalah "bukan aksara".

Data train/val/test memakai seed berbeda dan dibuat terpisah.
"""
import math
import random
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont

import config as C
from preprocessing import preprocess_gray

OUT = C.BASE_DIR / "outlier"
JUMLAH = {"train": 1500, "val": 300, "test": 300}
SEED = {"train": 101, "val": 202, "test": 303}
MAX_KORELASI = 0.60
SIZE = 400
import matplotlib                                   # font DejaVu ikut terpasang bersama matplotlib
_MPL = Path(matplotlib.get_data_path()) / "fonts" / "ttf"
FONTS = [str(p) for p in [_MPL / n for n in ("DejaVuSans.ttf", "DejaVuSans-Bold.ttf", "DejaVuSerif.ttf",
                                             "DejaVuSans-Oblique.ttf", "DejaVuSansMono.ttf",
                                             "DejaVuSerif-Italic.ttf")] if p.exists()]
HURUF = "abcdefghkmnopqrstuvwxyABCDEFHKMPQRSTVXY"
ANGKA = "04689"


def _spline(rng, n, spread=0.35):
    c = SIZE / 2
    pts = np.array([[c + rng.uniform(-spread, spread) * SIZE, c + rng.uniform(-spread, spread) * SIZE]
                    for _ in range(n)])
    t = np.linspace(0, 1, n); tt = np.linspace(0, 1, n * 25)
    return np.stack([np.interp(tt, t, pts[:, 0]), np.interp(tt, t, pts[:, 1])], 1)


def _smooth(path, k=9):
    if len(path) < k: return path
    ker = np.ones(k) / k
    return np.stack([np.convolve(path[:, i], ker, mode="valid") for i in range(2)], 1)


def coretan(d, rng, w):
    d.line([tuple(p) for p in _smooth(_spline(rng, rng.randint(5, 12)), k=45)], fill=0, width=w, joint="curve")


def ikal(d, rng, w):
    """Coretan berikal seperti tulisan sambung (trokoid)."""
    n = rng.randint(2, 6); t = np.linspace(0, 2 * math.pi * n, 500)
    r = rng.uniform(20, 55); a = rng.uniform(.8, 1.6) * r
    x = t / t.max() * rng.uniform(150, 300) + a * np.sin(t)
    y = r * np.cos(t) * rng.uniform(.8, 2.2) + rng.uniform(-40, 40) * np.sin(t / n)
    x += SIZE / 2 - (x.min() + x.max()) / 2; y += SIZE / 2 - (y.min() + y.max()) / 2
    if rng.random() < .5:
        x, y = y - y.mean() + SIZE / 2, x - x.mean() + SIZE / 2      # versi tegak
    d.line(list(zip(x, y)), fill=0, width=w, joint="curve")


def spiral(d, rng, w):
    c = SIZE / 2 + rng.uniform(-30, 30); turns = rng.uniform(1.5, 4); r0 = rng.uniform(60, 150)
    th = np.linspace(0, 2 * math.pi * turns, 400); r = r0 * th / th.max()
    d.line([(c + x, c + y) for x, y in zip(r * np.cos(th), r * np.sin(th))], fill=0, width=w, joint="curve")


def bentuk(d, rng, w):
    c = SIZE / 2; s = rng.uniform(70, 150); k = rng.choice(["lingkaran", "kotak", "segitiga", "bintang", "elips2"])
    if k == "lingkaran":
        d.ellipse([c - s, c - s, c + s, c + s], outline=0, width=w)
    elif k == "kotak":
        d.rectangle([c - s, c - s * .7, c + s, c + s * .7], outline=0, width=w)
    elif k == "segitiga":
        d.polygon([(c, c - s), (c - s, c + s * .8), (c + s, c + s * .8)], outline=0, width=w)
    elif k == "bintang":
        pts = [(c + (s if i % 2 == 0 else s * .45) * math.cos(i * math.pi / 5 - math.pi / 2),
                c + (s if i % 2 == 0 else s * .45) * math.sin(i * math.pi / 5 - math.pi / 2)) for i in range(10)]
        d.polygon(pts, outline=0, width=w)
    else:
        d.ellipse([c - s, c - s * .5, c + s, c + s * .5], outline=0, width=w)
        d.ellipse([c - s * .5, c - s, c + s * .5, c + s], outline=0, width=w)


def gelombang(d, rng, w):
    x = np.linspace(40, SIZE - 40, 300); f = rng.uniform(2, 6); a = rng.uniform(30, 90)
    if rng.random() < .5:
        y = SIZE / 2 + a * np.sin(x / SIZE * f * 2 * math.pi)
    else:
        y = SIZE / 2 + a * (2 * np.abs((x / SIZE * f) % 1 - .5) * 2 - 1)
    for k in range(rng.choice([1, 1, 2])):
        d.line(list(zip(x, y + k * rng.uniform(40, 80))), fill=0, width=w, joint="curve")


def arsiran(d, rng, w):
    n = rng.randint(6, 14); ang = rng.uniform(0, math.pi)
    for i in range(n):
        o = (i - n / 2) * rng.uniform(15, 25); dx, dy = math.cos(ang), math.sin(ang)
        cx, cy = SIZE / 2 - dy * o, SIZE / 2 + dx * o
        d.line([(cx - dx * 140, cy - dy * 140), (cx + dx * 140, cy + dy * 140)], fill=0, width=max(2, w // 2))


def teks(d, rng, w):
    if not FONTS:
        return coretan(d, rng, w)
    s = "".join(rng.choice(HURUF) for _ in range(rng.randint(2, 5))) if rng.random() < .75 \
        else "".join(rng.choice(ANGKA) for _ in range(rng.randint(2, 4)))
    size = rng.randint(90, 160)
    f = ImageFont.truetype(rng.choice(FONTS), size)
    bb = d.textbbox((0, 0), s, font=f)
    while bb[2] - bb[0] > SIZE * .9 and size > 40:            # pastikan teks muat
        size -= 10; f = ImageFont.truetype(f.path, size); bb = d.textbbox((0, 0), s, font=f)
    d.text(((SIZE - (bb[2] - bb[0])) / 2, (SIZE - (bb[3] - bb[1])) / 2 - bb[1]), s, font=f, fill=0,
           stroke_width=max(0, w // 4), stroke_fill=0)


JENIS = [(coretan, .22), (ikal, .18), (spiral, .08), (bentuk, .13), (gelombang, .09), (arsiran, .05), (teks, .25)]


def buat_satu(rng):
    img = Image.new("L", (SIZE, SIZE), 255); d = ImageDraw.Draw(img)
    w = rng.choice([3, 4, 5, 6, 8, 10, 12, 14])
    fn = rng.choices([j for j, _ in JENIS], weights=[p for _, p in JENIS])[0]
    fn(d, rng, w)
    if rng.random() < .2:                                    # campuran dua jenis
        rng.choices([j for j, _ in JENIS], weights=[p for _, p in JENIS])[0](d, rng, w)
    a = np.asarray(img.rotate(rng.uniform(-25, 25), fillcolor=255, resample=Image.BILINEAR), np.float32)
    if rng.random() < .4:
        # Gaya foto kamera: goresan sedikit kabur dan tinta tidak pekat. Latar tetap putih
        # dan tinta tetap gelap (0-15) seperti data aksara setelah preprocessing, agar model
        # tidak belajar jalan pintas "latar abu-abu = bukan aksara".
        ink = rng.uniform(0, 15)
        a = ink + (a / 255) * (255 - ink)
        a = cv2.GaussianBlur(a, (0, 0), rng.uniform(.6, 1.6))
    return np.clip(a, 0, 255).astype(np.uint8)


def vektor(g):
    v = cv2.resize(g, (48, 48), interpolation=cv2.INTER_AREA).ravel().astype(np.float32)
    v -= v.mean(); return v / (np.linalg.norm(v) + 1e-8)


def main():
    print("Memuat referensi aksara untuk penyaringan kemiripan ...")
    ref = []
    for s in ("train", "val", "test"):
        df = pd.read_csv(C.OUT_DIR / f"split_{s}.csv")
        ref += [vektor(preprocess_gray(C.DATASET_DIR / r)) for r in df.relpath]
    ref = np.stack(ref)
    tmp = OUT / "_tmp.png"; OUT.mkdir(exist_ok=True)
    for split, n in JUMLAH.items():
        d = OUT / split; d.mkdir(parents=True, exist_ok=True)
        for f in d.glob("*.png"): f.unlink()
        rng = random.Random(SEED[split]); k = dibuang = 0
        while k < n:
            a = buat_satu(rng)
            cv2.imwrite(str(tmp), a)
            g = preprocess_gray(tmp)
            if float((ref @ vektor(g)).max()) >= MAX_KORELASI:
                dibuang += 1; continue
            cv2.imwrite(str(d / f"outlier_{split}_{k:04d}.png"), a); k += 1
        print(f"{split}: {n} citra (dibuang karena mirip aksara: {dibuang})")
    tmp.unlink(missing_ok=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile train_oe.py
"""M2-OE: M2 (ResNet50-GLCM) yang dilatih lanjut dengan Outlier Exposure.

    python buat_outlier.py     # sekali saja: membuat contoh coretan non-aksara di outlier/
    python train_oe.py         # butuh hasil_M2/best.keras (latih M2 dulu)

Masalah yang diatasi
    Model M1-M4 hanya mengenal 30 kelas (closed-set). Coretan yang bukan aksara
    tetap dipaksa masuk ke salah satu kelas, sering dengan persentase tinggi.

Cara kerja (Hendrycks dkk., 2019)
    Model M2 dilatih lanjut dengan dua jenis data dalam setiap batch:
      - citra aksara  -> target one-hot kelas sebenarnya (seperti biasa)
      - citra coretan -> target seragam 1/30 untuk semua kelas
    Loss = CE(aksara) + LAMBDA * CE(coretan, seragam)
    Akibatnya, untuk coretan model belajar memberi persentase kecil yang merata
    (sekitar 3% per kelas), sementara aksara tetap diberi persentase tinggi.
    Softmax dan 30 kelas tetap sama; tidak ada kelas "bukan aksara".

Ambang "tidak dikenali" ditentukan dari data validasi aksara: persentil ke-5
probabilitas tertinggi, sehingga 95% aksara asli tetap diterima (standar FPR95).

Hasil di hasil_M2OE/:
    best.keras, history.csv, kurva_pelatihan.png, ambang_ood.json,
    ringkasan_hasil.json, tabel_oe.csv, grafik_oe.png, contoh_coretan_oe.png
M1-M4 dan hasil_M2/ tidak diubah.
"""
import argparse
import json
import math
import time

import cv2
import joblib
import keras
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from keras import ops
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

import config as C
from augmentation import augment_gray
from data_loader import load_cache, load_split
from glcm_features import glcm_features
from preprocessing import gray_to_rgb, normalize_for, preprocess_gray

try:
    from keras.utils import PyDataset as _Base
except ImportError:
    from tensorflow.keras.utils import Sequence as _Base

ASAL, ARCH, GLCM = "M2", "resnet50", True
OUT = C.BASE_DIR / "hasil_M2OE"
OUTLIER_DIR = C.BASE_DIR / "outlier"
P = dict(lambda_oe=0.5, batch_aksara=32, batch_coretan=32,
         lr_tahap1=1e-4, lr_tahap2=1e-5, epochs_tahap1=10, epochs_tahap2=10,
         unfreeze_last=30, patience=3, persentil_ambang=5)


# ---------------------------------------------------------------- data
def muat_coretan(split):
    """Citra coretan setelah preprocessing yang sama dengan aksara (disimpan sebagai cache)."""
    cache = OUTLIER_DIR / f"outlier_{split}_224.npz"
    files = sorted((OUTLIER_DIR / split).glob("*.png"))
    if not files:
        raise SystemExit(f"{OUTLIER_DIR / split} kosong. Jalankan dulu: python buat_outlier.py")
    if cache.exists():
        z = np.load(cache)
        if len(z["images"]) == len(files):
            return list(z["images"])
    print(f"Preprocessing {len(files)} citra coretan ({split}) ...")
    g = np.stack([preprocess_gray(f) for f in files])
    np.savez_compressed(cache, images=g)
    return list(g)


def muat_aksara(split, cache, idx):
    df = load_split(split)
    grays = [cache[r] if r in cache else preprocess_gray(C.DATASET_DIR / r) for r in df.relpath]
    return grays, np.array([idx[l] for l in df.label], np.int32)


def ke_input(grays, scaler):
    x = normalize_for(ARCH, np.stack([gray_to_rgb(g) for g in grays]))
    if not GLCM:
        return x
    f = scaler.transform(np.stack([glcm_features(g) for g in grays])).astype(np.float32)
    return (x, f)


class OEDataset(_Base):
    """Setiap batch: aksara (target one-hot, bobot 1) + coretan (target 1/30, bobot LAMBDA)."""

    def __init__(self, ak_grays, ak_y, co_grays, n_kelas, scaler, augment=False, shuffle=False,
                 seed=C.SEED, **kw):
        super().__init__(**kw)
        self.ak, self.y, self.co = ak_grays, ak_y, co_grays
        self.n, self.scaler, self.augment, self.shuffle = n_kelas, scaler, augment, shuffle
        self.rng = np.random.default_rng(seed)
        self.o_ak, self.o_co = np.arange(len(ak_grays)), np.arange(len(co_grays))
        self.k_co = P["batch_coretan"] if shuffle else math.ceil(len(co_grays) / len(self))
        self.on_epoch_end()

    def __len__(self):
        return math.ceil(len(self.ak) / P["batch_aksara"])

    def on_epoch_end(self):
        if self.shuffle:
            self.rng.shuffle(self.o_ak); self.rng.shuffle(self.o_co)

    def __getitem__(self, b):
        ia = self.o_ak[b * P["batch_aksara"]:(b + 1) * P["batch_aksara"]]
        ic = np.take(self.o_co, range(b * self.k_co, (b + 1) * self.k_co), mode="wrap")
        if not self.shuffle:                                  # validasi: tiap coretan dipakai sekali
            ic = self.o_co[b * self.k_co:(b + 1) * self.k_co]
        grays = [self.ak[i] for i in ia] + [self.co[i] for i in ic]
        if self.augment:
            grays = [augment_gray(g, self.rng) for g in grays]
        y = np.concatenate([np.eye(self.n, dtype=np.float32)[self.y[ia]],
                            np.full((len(ic), self.n), 1 / self.n, np.float32)])
        w = np.concatenate([np.ones(len(ia), np.float32),
                            np.full(len(ic), P["lambda_oe"], np.float32)])
        return ke_input(grays, self.scaler), y, w


class AkurasiAksara(keras.metrics.Metric):
    """Akurasi yang hanya dihitung pada citra aksara (coretan tidak punya kelas benar)."""

    def __init__(self, name="akurasi_aksara", **kw):
        super().__init__(name=name, **kw)
        self.benar = self.add_variable(shape=(), initializer="zeros", name="benar")
        self.total = self.add_variable(shape=(), initializer="zeros", name="total")

    def update_state(self, y_true, y_pred, sample_weight=None):
        m = ops.cast(ops.max(y_true, axis=-1) > 0.5, "float32")
        c = ops.cast(ops.argmax(y_true, axis=-1) == ops.argmax(y_pred, axis=-1), "float32")
        self.benar.assign(self.benar + ops.sum(c * m)); self.total.assign(self.total + ops.sum(m))

    def result(self):
        return self.benar / ops.maximum(self.total, 1.0)

    def reset_state(self):
        self.benar.assign(0.0); self.total.assign(0.0)


# ---------------------------------------------------------------- evaluasi
def prob(model, grays, scaler, bs=32):
    return np.concatenate([model.predict(ke_input(grays[i:i + bs], scaler), verbose=0)
                           for i in range(0, len(grays), bs)])


def nilai(nama, model, data, scaler):
    pv, pt, pc = (prob(model, data[k], scaler) for k in ("val_ak", "test_ak", "test_co"))
    yt = data["test_y"]
    ambang = float(np.percentile(pv.max(1), P["persentil_ambang"]))
    pred = pt.argmax(1)
    return dict(model=nama,
                akurasi_uji_aksara=round(accuracy_score(yt, pred), 4),
                f1_macro_uji_aksara=round(f1_score(yt, pred, average="macro"), 4),
                ambang_probabilitas=round(ambang, 4),
                aksara_uji_diterima=round(float((pt.max(1) >= ambang).mean()), 4),
                coretan_uji_ditolak=round(float((pc.max(1) < ambang).mean()), 4),
                auroc=round(roc_auc_score(np.r_[np.ones(len(pt)), np.zeros(len(pc))],
                                          np.r_[pt.max(1), pc.max(1)]), 4),
                rerata_prob_tertinggi_aksara=round(float(pt.max(1).mean()), 4),
                rerata_prob_tertinggi_coretan=round(float(pc.max(1).mean()), 4)), pt, pc


def grafik(hasil, path):
    fig, axs = plt.subplots(1, len(hasil), figsize=(5.5 * len(hasil), 3.8), squeeze=False)
    bins = np.linspace(0, 1, 41)
    for ax, (r, pt, pc) in zip(axs[0], hasil):
        ax.hist(pt.max(1), bins, alpha=.7, label="aksara (uji)", color="#2b6cb0")
        ax.hist(pc.max(1), bins, alpha=.7, label="coretan (uji)", color="#dd6b20")
        ax.axvline(r["ambang_probabilitas"], color="k", ls="--", lw=1)
        ax.set(title=f"{r['model']}  (AUROC {r['auroc']:.3f})", yscale="log",
               xlabel="Probabilitas tertinggi (softmax)", ylabel="Jumlah citra (log)")
        ax.legend(fontsize=8)
    fig.tight_layout(); fig.savefig(path, dpi=130); plt.close(fig)


def contoh_coretan(model, grays, scaler, classes, path, n=16):
    p = prob(model, grays[:n], scaler)
    fig, axs = plt.subplots(2, n // 2, figsize=(n * .95, 4.4))
    for ax, g, q in zip(axs.ravel(), grays, p):
        ax.imshow(g, cmap="gray", vmin=0, vmax=255); ax.axis("off")
        ax.set_title(f"{classes[q.argmax()]} {q.max() * 100:.0f}%", fontsize=9)
    fig.suptitle("M2-OE pada coretan uji (kelas teratas dan probabilitasnya)")
    fig.tight_layout(); fig.savefig(path, dpi=110); plt.close(fig)


# ---------------------------------------------------------------- utama
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--smoke", action="store_true", help="uji cepat dengan data sedikit")
    a = ap.parse_args()
    keras.utils.set_random_seed(C.SEED)
    OUT.mkdir(exist_ok=True)
    src = C.BASE_DIR / f"hasil_{ASAL}" / "best.keras"
    if not src.exists():
        raise SystemExit(f"{src} tidak ditemukan. Latih dulu: python train_model.py --model {ASAL}")

    classes = json.load(open(C.OUT_DIR / "class_names.json"))
    idx = {c: i for i, c in enumerate(classes)}
    scaler = joblib.load(C.OUT_DIR / "scaler_glcm.joblib")
    cache = load_cache() if (C.OUT_DIR / "preprocessed_224.npz").exists() else {}
    d = {}
    for s in ("train", "val", "test"):
        d[f"{s}_ak"], d[f"{s}_y"] = muat_aksara(s, cache, idx)
        d[f"{s}_co"] = muat_coretan(s)
    if a.smoke:
        for k in d:
            d[k] = d[k][:64]
        P.update(epochs_tahap1=1, epochs_tahap2=1)

    train = OEDataset(d["train_ak"], d["train_y"], d["train_co"], len(classes), scaler,
                      augment=True, shuffle=True)
    val = OEDataset(d["val_ak"], d["val_y"], d["val_co"], len(classes), scaler)

    model = keras.models.load_model(src, compile=False)
    base = next(l for l in model.layers if isinstance(l, keras.Model))
    wpath = str(OUT / "best.weights.h5")
    ckpt = keras.callbacks.ModelCheckpoint(wpath, monitor="val_loss", save_best_only=True,
                                           save_weights_only=True)

    def latih(lr, epochs):
        model.compile(keras.optimizers.Adam(lr), loss="categorical_crossentropy",
                      metrics=[AkurasiAksara()])
        es = keras.callbacks.EarlyStopping("val_loss", patience=P["patience"], restore_best_weights=True)
        return model.fit(train, validation_data=val, epochs=epochs, callbacks=[es, ckpt])

    t0 = time.time()
    print("== M2-OE | Tahap 1: kepala klasifikasi dilatih dengan coretan (backbone dibekukan) ==")
    base.trainable = False
    h1 = latih(P["lr_tahap1"], P["epochs_tahap1"])
    print(f"== Tahap 2: fine-tuning {P['unfreeze_last']} lapisan akhir backbone ==")
    base.trainable = True
    for layer in base.layers[:-P["unfreeze_last"]]:
        layer.trainable = False
    h2 = latih(P["lr_tahap2"], P["epochs_tahap2"])
    menit = (time.time() - t0) / 60

    model.load_weights(wpath)
    model.compile(loss="categorical_crossentropy")      # simpan tanpa metrik khusus agar mudah dimuat
    model.save(OUT / "best.keras")
    (OUT / "best.weights.h5").unlink(missing_ok=True)

    hist = pd.concat([pd.DataFrame(h1.history).assign(tahap=1),
                      pd.DataFrame(h2.history).assign(tahap=2)], ignore_index=True)
    hist.insert(0, "epoch", np.arange(1, len(hist) + 1))
    hist.to_csv(OUT / "history.csv", index=False)
    fig, axs = plt.subplots(1, 2, figsize=(11, 4))
    for ax, m, lab in [(axs[0], "loss", "Loss (aksara + coretan)"), (axs[1], "akurasi_aksara", "Akurasi aksara")]:
        ax.plot(hist.epoch, hist[m], label="train"); ax.plot(hist.epoch, hist[f"val_{m}"], label="validasi")
        ax.axvline(len(h1.history["loss"]) + .5, color="gray", ls="--", lw=1)
        ax.set(xlabel="Epoch", title=lab); ax.grid(alpha=.3); ax.legend()
    fig.suptitle("Kurva pelatihan M2-OE"); fig.tight_layout(); fig.savefig(OUT / "kurva_pelatihan.png", dpi=130)
    plt.close(fig)

    print("== Evaluasi: M2 (tanpa OE) vs M2-OE ==")
    hasil = [nilai("M2", keras.models.load_model(src, compile=False), d, scaler),
             nilai("M2-OE", model, d, scaler)]
    tabel = pd.DataFrame([h[0] for h in hasil])
    tabel.to_csv(OUT / "tabel_oe.csv", index=False)
    print(tabel.T.to_string(header=False))
    grafik(hasil, OUT / "grafik_oe.png")
    contoh_coretan(model, d["test_co"], scaler, classes, OUT / "contoh_coretan_oe.png")

    r = hasil[1][0]
    json.dump({"ambang": r["ambang_probabilitas"], "persentil_validasi_aksara": P["persentil_ambang"]},
              open(OUT / "ambang_ood.json", "w"), indent=1)
    json.dump({"model": "M2OE", "nama": "ResNet50-GLCM + Outlier Exposure", "arsitektur": ARCH,
               "glcm": GLCM, "dilanjutkan_dari": ASAL,
               "akurasi_uji": r["akurasi_uji_aksara"], "f1_macro": r["f1_macro_uji_aksara"],
               "jumlah_coretan": {s: len(d[f"{s}_co"]) for s in ("train", "val", "test")},
               "evaluasi_ood": r, "epoch_total": int(len(hist)),
               "epoch_tahap1": len(h1.history["loss"]), "waktu_latih_menit": round(menit, 1),
               "seed": C.SEED, "parameter": P},
              open(OUT / "ringkasan_hasil.json", "w"), indent=1)
    print(f"Selesai. Hasil di {OUT}")


if __name__ == "__main__":
    main()


## 3. Buat contoh coretan non-aksara
1.500 latih, 300 validasi, 300 uji. Coretan yang terlalu mirip aksara asli dibuang otomatis.

In [ ]:
if os.path.exists('outlier/test') and len(os.listdir('outlier/test')) >= 300:
    print('Folder outlier/ sudah ada, dilewati.')
else:
    !python buat_outlier.py

In [ ]:
import glob, random, matplotlib.pyplot as plt
from preprocessing import preprocess_gray
fs = sorted(glob.glob('outlier/train/*.png')); random.seed(3)
fig, ax = plt.subplots(3, 8, figsize=(14, 5.5))
for a, p in zip(ax.ravel(), random.sample(fs, 24)):
    a.imshow(preprocess_gray(p), cmap='gray', vmin=0, vmax=255); a.axis('off')
fig.suptitle('Contoh coretan non-aksara (setelah preprocessing)'); plt.show()

## 4. Latih M2-OE dan bandingkan dengan M2

In [ ]:
!python train_oe.py

## 5. Tampilkan hasil

In [ ]:
import pandas as pd, json
from IPython.display import Image, display
display(pd.read_csv('hasil_M2OE/tabel_oe.csv').set_index('model').T)
print('Ambang aplikasi:', json.load(open('hasil_M2OE/ambang_ood.json')))
for f in ['grafik_oe.png', 'contoh_coretan_oe.png', 'kurva_pelatihan.png']:
    display(Image(f'hasil_M2OE/{f}'))

## 6. Unduh hasil_M2OE ke laptop
Ekstrak zip ini ke folder proyek di laptop sehingga ada `hasil_M2OE/best.keras` dan `hasil_M2OE/ambang_ood.json`. Aplikasi otomatis memakai M2-OE bila folder ini ada.

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('/content/hasil_M2OE', 'zip', '.', 'hasil_M2OE')
files.download('/content/hasil_M2OE.zip')